# Process ensemble summary

This notebook takes `ensemble_summary_cox_xgb.csv` and produces a per-disease table with:
- best base-model test AUC per time frame (1/2/5/10 years), formatted as `0.988 (xgb)`
- best ensemble test AUC per time frame
- external AUC of the base model that gave the best test AUC per time frame
- external AUC of the ensemble method that gave the best test AUC per time frame.

In [23]:
import os
import numpy as np
import pandas as pd

SUMMARY_PATH = "ensemble_cancer_summary.csv" 
OUTPUT_PATH = "ensemble_summary_by_disease.csv"
BASE_MODELS = ["cox", "xgb", "ordinal", "rsf"]

TIME_FRAMES = [5]

print("Summary path:", SUMMARY_PATH)
print("Output path:", OUTPUT_PATH)

Summary path: ensemble_cancer_summary.csv
Output path: ensemble_summary_by_disease.csv


In [30]:
summary_df.loc[summary_df['disease']=='uterine_cancer']

,disease,time_frame,ensemble_method,valid_AUC,test_AUC,external_AUC,n_valid,n_test,n_ext
45,uterine_cancer,5,cox,0.451165,0.673148,NaN,1120,5690,0
46,uterine_cancer,5,xgb,0.453629,0.722917,NaN,1120,5690,0
47,uterine_cancer,5,rsf,0.228495,0.695424,NaN,1120,5690,0
48,uterine_cancer,5,weighted_softmax_ensemble,0.405466,0.725776,NaN,1120,5690,0
49,uterine_cancer,5,pnn_ensemble,0.725470,0.300037,NaN,1120,5690,0


In [24]:
# Load summary CSV

summary_df = pd.read_csv(SUMMARY_PATH)
print(summary_df.head())
print("\nUnique ensemble_methods:", summary_df["ensemble_method"].unique())
print("Unique time_frames:", sorted(summary_df["time_frame"].unique()))

         disease  time_frame            ensemble_method  valid_AUC  test_AUC  \
0  breast_cancer           5                        cox   0.546454  0.571571   
1  breast_cancer           5                        xgb   0.440559  0.480863   
2  breast_cancer           5                        rsf   0.455330  0.500961   
3  breast_cancer           5  weighted_softmax_ensemble   0.535036  0.560819   
4  breast_cancer           5               pnn_ensemble   0.537570  0.530574   

   external_AUC  n_valid  n_test  n_ext  
0           NaN     1091    5540      0  
1           NaN     1091    5540      0  
2           NaN     1091    5540      0  
3           NaN     1091    5540      0  
4           NaN     1091    5540      0  

Unique ensemble_methods: ['cox' 'xgb' 'rsf' 'weighted_softmax_ensemble' 'pnn_ensemble']
Unique time_frames: [5]


In [28]:
def format_auc_with_model(value: float, model: str | None) -> str:
    """Format AUC as '0.988 (xgb)'. Returns empty string if value is NaN or model is None."""
    if model is None or pd.isna(value):
        return ""
    if model == "pnn_ensemble":
        model = "pnn"
    elif model == "weighted_softmax_ensemble":
        model = "weighted"
    return f"{value:.3f} ({model})"

In [12]:
# Build per-disease table

rows = []

for disease, df_d in summary_df.groupby("disease"):
    row = {"disease": disease}
    for tf in TIME_FRAMES:
        df_tf = df_d[df_d["time_frame"] == tf]
        if df_tf.empty:
            # No data for this (disease, tf)
            row[f"test_auc_{tf}yr"] = ""
            row[f"ensemble_test_auc_{tf}yr"] = np.nan
            row[f"external_auc_{tf}yr"] = np.nan
            row[f"ensemble_external_auc_{tf}yr"] = np.nan
            continue

        # --- Base models: pick the one with best test AUC ---
        base_df = df_tf[df_tf["ensemble_method"].isin(BASE_MODELS)].copy()
        if base_df.empty or base_df["test_AUC"].dropna().empty:
            best_base_test_str = ""
            best_base_ext = np.nan
        else:
            # idx of max test_AUC among base models
            idx_best = base_df["test_AUC"].idxmax()
            best_row = base_df.loc[idx_best]
            best_base_test_str = format_auc_with_model(best_row["test_AUC"], best_row["ensemble_method"])
            best_base_ext = best_row["external_AUC"]

        # --- Ensemble methods: choose method with best test AUC ---
        ens_df = df_tf[~df_tf["ensemble_method"].isin(BASE_MODELS)].copy()
        if ens_df.empty or ens_df["test_AUC"].dropna().empty:
            best_ens_test = np.nan
            best_ens_ext = np.nan
        else:
            idx_best_ens = ens_df["test_AUC"].idxmax()
            best_ens_row = ens_df.loc[idx_best_ens]
            best_ens_test = format_auc_with_model(best_ens_row["test_AUC"], best_ens_row["ensemble_method"])
            best_ens_ext = best_ens_row["external_AUC"]

        row[f"test_auc_{tf}yr"] = best_base_test_str
        row[f"ensemble_test_auc_{tf}yr"] = best_ens_test
        # row[f"external_auc_{tf}yr"] = best_base_ext
        # row[f"ensemble_external_auc_{tf}yr"] = best_ens_ext

    rows.append(row)

wide_df = pd.DataFrame(rows)
wide_df

,disease,test_auc_5yr,ensemble_test_auc_5yr
0,breast_cancer,0.572 (cox),0.561 (weighted)
1,colorectal_cancer,0.618 (xgb),0.604 (weighted)
2,kidney_cancer,0.867 (cox),0.815 (weighted)
3,lung_cancer,0.876 (xgb),0.881 (pnn)
4,lymphoma_cancer,0.852 (xgb),0.816 (weighted)
5,melanoma_cancer,0.537 (cox),0.526 (pnn)
6,ovarian_cancer,0.671 (rsf),0.686 (weighted)
7,prostate_cancer,0.804 (xgb),0.814 (weighted)
8,skin_cancer,0.655 (xgb),0.661 (weighted)
9,uterine_cancer,0.723 (xgb),0.726 (weighted)


In [ ]:
# Reorder columns: all test AUC columns first, then external AUC columns

test_cols = []
external_cols = []
for tf in TIME_FRAMES:
    test_cols.extend([
        f"test_auc_{tf}yr",
        f"ensemble_test_auc_{tf}yr",
    ])
    external_cols.extend([
        f"external_auc_{tf}yr",
        f"ensemble_external_auc_{tf}yr",
    ])

ordered_cols = ["disease"] + test_cols + external_cols
# Keep only columns that actually exist (in case some combos were missing)
ordered_cols = [c for c in ordered_cols if c in wide_df.columns]

wide_df = wide_df[ordered_cols]
print("Column order:", list(wide_df.columns))

In [ ]:
# Save to CSV

wide_df.to_csv(OUTPUT_PATH, index=False)
print("Wrote:", OUTPUT_PATH)
wide_df.head()

In [5]:
# Analyze which ensemble method is best most often and its average best test AUC

# Define ensemble methods (exclude base models)
ensemble_methods = sorted(
    m for m in summary_df["ensemble_method"].unique() if m not in BASE_MODELS
)
print("Ensemble methods considered:", ensemble_methods)

best_rows = []

# For each (disease, time_frame), find the ensemble method with max test_AUC
for (disease, tf), g in summary_df.groupby(["disease", "time_frame"]):
    g_ens = g[g["ensemble_method"].isin(ensemble_methods)].dropna(subset=["test_AUC"])
    if g_ens.empty:
        continue
    idx_best = g_ens["test_AUC"].idxmax()
    best_rows.append(g_ens.loc[idx_best])

best_df = pd.DataFrame(best_rows)
print(f"Total (disease, time_frame) combos with at least one ensemble model: {len(best_df)}")

# Aggregate: how often each ensemble method is best, and its mean best test AUC
stats = (
    best_df.groupby("ensemble_method")["test_AUC"]
    .agg(count="count", mean_test_AUC="mean")
    .reset_index()
)

stats["percent_best"] = 100.0 * stats["count"] / stats["count"].sum()

# Order by percent_best descending
stats = stats.sort_values("percent_best", ascending=False)

stats

Ensemble methods considered: ['pnn_ensemble', 'weighted_softmax_ensemble']
Total (disease, time_frame) combos with at least one ensemble model: 10


,ensemble_method,count,mean_test_AUC,percent_best
1,weighted_softmax_ensemble,8,0.710259,80.0
0,pnn_ensemble,2,0.703561,20.0


In [5]:
# Save to CSV

wide_df.to_csv(OUTPUT_PATH, index=False)
print("Wrote:", OUTPUT_PATH)
wide_df.head()

Wrote: ensemble_summary_by_disease_cox_xgb.csv


,disease,test_auc_1yr,ensemble_test_auc_1yr,external_auc_1yr,ensemble_external_auc_1yr,test_auc_2yr,ensemble_test_auc_2yr,external_auc_2yr,ensemble_external_auc_2yr,test_auc_5yr,ensemble_test_auc_5yr,external_auc_5yr,ensemble_external_auc_5yr,test_auc_10yr,ensemble_test_auc_10yr,external_auc_10yr,ensemble_external_auc_10yr
0,acute_kidney_injury,0.867 (cox),0.865328,0.986806,0.985778,0.907 (cox),0.906916,0.739696,0.737038,0.873 (cox),0.877273,0.787398,0.792878,0.830 (xgb),0.836908,0.832424,0.828300
1,alzheimers_disease,,NaN,NaN,NaN,0.775 (cox),0.771056,0.860417,0.871952,0.846 (cox),0.863755,0.830978,0.907360,0.864 (xgb),0.870527,0.897773,0.908017
2,atrial_fibrillation,0.913 (cox),0.906754,0.862184,0.861482,0.842 (cox),0.842367,0.750006,0.761802,0.809 (cox),0.814345,0.785201,0.785465,0.788 (xgb),0.797018,0.760472,0.769473
3,chronic_kidney_disease,0.903 (cox),0.904781,0.989700,0.992103,0.866 (cox),0.872914,0.992142,0.992314,0.881 (cox),0.887993,0.966346,0.969509,0.872 (xgb),0.875034,0.897645,0.902376
4,copd,0.772 (xgb),0.765770,0.748134,0.806176,0.830 (cox),0.842955,0.850469,0.845149,0.842 (xgb),0.844854,0.855517,0.861247,0.826 (xgb),0.833564,0.849133,0.856507


## Get PNN probabilities

In [30]:
# Rerun PNN and save test/external predictions for a specific (disease, time_frame)

from run_ensemble_disease_risk_cox_xgb import (
    DATA_PATH as SCRIPT_DATA_PATH,
    MODELS,
    load_probs_for_split,
    filter_for_future_prediction,
    get_labels_future,
)
from models import pnn_ensemble
import numpy as np


TIME_FRAMES = [1, 2, 5, 10]

DISEASES = [
    "acute_kidney_injury",
    "alzheimers_disease",
    "atrial_fibrillation",
    "chronic_kidney_disease",
    "copd",
    "end_stage_renal_disease",
    "heart_failure",
    "hypertensive_heart_kidney_diseases",
    "ischemic_heart_disease",
    "liver_disease",
    "lower_respiratory_disease",
    "other_dementia",
    "parkinsons",
    "peripheral_vascular_disease",
    "stroke",
    "type_1_diabetes",
    "type_2_diabetes",
]

# Optional: fix seeds for *this* rerun (does NOT reproduce an old, unseeded run)
# import torch
# np.random.seed(0)
# torch.manual_seed(0)

# ---- Load label data ----
data_path = SCRIPT_DATA_PATH
labels_path = os.path.dirname(data_path)

valid_df_r = pd.read_csv(os.path.join(labels_path, "ukb_disease_valid.csv"), low_memory=False)
test_df_r = pd.read_csv(os.path.join(labels_path, "ukb_disease_test.csv"), low_memory=False)
external_path_r = os.path.join(labels_path, "ukb_disease_scotland_wales.csv")
external_df_r = pd.read_csv(external_path_r, low_memory=False) if os.path.isfile(external_path_r) else None

def rerun_pnn(rerun_disease, rerun_time_frame):
    # ---- Match script logic: filter + labels ----
    valid_f = filter_for_future_prediction(valid_df_r, rerun_disease)
    test_f = filter_for_future_prediction(test_df_r, rerun_disease)
    external_f = filter_for_future_prediction(external_df_r, rerun_disease) if external_df_r is not None else None

    y_valid = get_labels_future(valid_f, rerun_disease, rerun_time_frame)
    y_test = get_labels_future(test_f, rerun_disease, rerun_time_frame)
    y_ext = get_labels_future(external_f, rerun_disease, rerun_time_frame) if external_f is not None else None

    if y_valid is None or y_test is None:
        print(f"No labels for {rerun_disease} @ {rerun_time_frame}yr; skipping.")
        return

    # ---- Load probabilities (same as script) ----
    X_valid_probs = load_probs_for_split(data_path, rerun_disease, rerun_time_frame, "valid")
    X_test_probs = load_probs_for_split(data_path, rerun_disease, rerun_time_frame, "test")
    X_ext_probs = load_probs_for_split(data_path, rerun_disease, rerun_time_frame, "external") if external_df_r is not None else None

    if X_valid_probs is None or X_test_probs is None:
        print(f"No prob files for {rerun_disease} @ {rerun_time_frame}yr; skipping.")
        return

    # ---- Merge on eid and align X/y (same as run_one) ----
    valid_merge = X_valid_probs[["eid"]].merge(
        valid_f[["eid"]].assign(y=y_valid.values), on="eid", how="inner"
    )
    test_merge = X_test_probs[["eid"]].merge(
        test_f[["eid"]].assign(y=y_test.values), on="eid", how="inner"
    )
    ext_merge = None
    if X_ext_probs is not None and y_ext is not None and external_f is not None:
        ext_merge = X_ext_probs[["eid"]].merge(
            external_f[["eid"]].assign(y=y_ext.values), on="eid", how="inner"
        )

    y_train = valid_merge["y"].values
    y_test_arr = test_merge["y"].values
    y_ext_arr = ext_merge["y"].values if ext_merge is not None else None

    train_eids = valid_merge["eid"].values
    test_eids = test_merge["eid"].values
    ext_eids = ext_merge["eid"].values if ext_merge is not None else None

    X_train_probs = X_valid_probs.set_index("eid").loc[train_eids][MODELS].values
    X_test_probs_arr = X_test_probs.set_index("eid").loc[test_eids][MODELS].values
    X_ext_probs_arr = None
    if ext_eids is not None and X_ext_probs is not None:
        X_ext_probs_arr = X_ext_probs.set_index("eid").loc[ext_eids][MODELS].values

    # ---- Complete-case filter (same as script), also track eids ----
    if np.isnan(X_train_probs).all() or np.isnan(X_test_probs_arr).all():
        print(f"Skipping {rerun_disease} @ {rerun_time_frame}yr: all probabilities are NaN")
        return

    train_ok = ~np.isnan(X_train_probs).any(axis=1)
    test_ok = ~np.isnan(X_test_probs_arr).any(axis=1)
    if train_ok.sum() == 0 or test_ok.sum() == 0:
        print(f"Skipping {rerun_disease} @ {rerun_time_frame}yr: no complete-case rows")
        return

    X_train_probs = X_train_probs[train_ok].astype(np.float64)
    y_train = y_train[train_ok]
    train_eids_f = train_eids[train_ok]

    X_test_probs_arr = X_test_probs_arr[test_ok].astype(np.float64)
    y_test_arr = y_test_arr[test_ok]
    test_eids_f = test_eids[test_ok]

    ext_eids_f = None
    if X_ext_probs_arr is not None and y_ext_arr is not None:
        ext_ok = ~np.isnan(X_ext_probs_arr).any(axis=1)
        X_ext_probs_arr = X_ext_probs_arr[ext_ok].astype(np.float64)
        y_ext_arr = y_ext_arr[ext_ok]
        ext_eids_f = ext_eids[ext_ok]

    n_pos_train = int(y_train.sum())
    n_pos_test = int(y_test_arr.sum())
    if n_pos_train < 2 or n_pos_test < 2:
        print(f"Skipping {rerun_disease} @ {rerun_time_frame}yr: too few positives")
        return

    # ---- Run PNN ensemble (same call as script) ----
    res_pnn = pnn_ensemble(
        X_train_probs, y_train,
        X_test_probs_arr, y_test_arr,
        X_ext_probs=X_ext_probs_arr,
        y_ext=y_ext_arr,
    )

    # ---- Build and save prediction CSVs ----
    # Test set
    test_pred_proba = res_pnn.proba_test[:, 1]  # positive-class prob
    test_out_df = pd.DataFrame({
        "eid": test_eids_f,
        "y_true": y_test_arr,
        "y_pred_proba": test_pred_proba,
    })
    
    output_dir = f"{labels_path}/disease_risk/pnn/"
    test_fname = f"{output_dir}pnn_{rerun_disease}_{rerun_time_frame}yr_test_preds.csv"
    test_out_df.to_csv(test_fname, index=False)
    print("Wrote test preds to", test_fname)

    # External set (if available)
    if res_pnn.proba_ext is not None and res_pnn.y_ext is not None and ext_eids_f is not None:
        ext_pred_proba = res_pnn.proba_ext[:, 1]
        ext_out_df = pd.DataFrame({
            "eid": ext_eids_f,
            "y_true": res_pnn.y_ext,
            "y_pred_proba": ext_pred_proba,
        })
        ext_fname = f"{output_dir}pnn_{rerun_disease}_{rerun_time_frame}yr_external_preds.csv"
        ext_out_df.to_csv(ext_fname, index=False)
        print("Wrote external preds to", ext_fname)
    else:
        print("No external predictions available for this combo.")
    return

In [ ]:
for disease in DISEASES:
    for time_frame in TIME_FRAMES:
        rerun_pnn(disease, time_frame)

Wrote test preds to /orcd/pool/003/dbertsim_shared/ukb/disease_risk/pnn/pnn_acute_kidney_injury_1yr_test_preds.csv
Wrote external preds to /orcd/pool/003/dbertsim_shared/ukb/disease_risk/pnn/pnn_acute_kidney_injury_1yr_external_preds.csv
